In [1]:
import time
from pathlib import Path

import ctranslate2
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm
from transformers import AutoTokenizer

# ==============================
# Minimal + Fast NLLB -> CT2 Demo (CPU, ohne lokale Konvertierung)
# ==============================
CT2_REPO_ID = "JustFrederik/nllb-200-distilled-600M-ct2-int8"
CT2_MODEL_DIR = Path("data/models/nllb-200-distilled-600M-ct2-int8")
TOKENIZER_ID = "facebook/nllb-200-distilled-600M"
TARGET_LANG = "eng_Latn"

# Ryzen 7 PRO 5850U (8C/16T): guter Throughput-Startpunkt
INTER_THREADS = 4
INTRA_THREADS = 4
BATCH_SIZE = 64
MAX_BATCH_TOKENS = 4096


def _is_ct2_model_ready(model_dir: Path) -> bool:
    required = ["model.bin", "config.json", "tokenizer.json", "sentencepiece.bpe.model"]
    return model_dir.is_dir() and all((model_dir / name).exists() for name in required)


def ensure_ct2_model() -> Path:
    if _is_ct2_model_ready(CT2_MODEL_DIR):
        print(f"CT2-Modell bereits vorhanden: {CT2_MODEL_DIR}")
        return CT2_MODEL_DIR

    print("Lade vor-konvertiertes CTranslate2-Modell (einmalig)...")
    t0 = time.time()
    snapshot_download(
        repo_id=CT2_REPO_ID,
        local_dir=str(CT2_MODEL_DIR),
        max_workers=8,
    )
    if not _is_ct2_model_ready(CT2_MODEL_DIR):
        raise RuntimeError("CT2-Modelldownload unvollstaendig.")

    print(f"✅ Download abgeschlossen in {time.time() - t0:.1f}s")
    return CT2_MODEL_DIR


def encode_nllb(text: str, src_lang: str, tokenizer: AutoTokenizer):
    tokenizer.src_lang = src_lang
    token_ids = tokenizer.encode(text)
    return tokenizer.convert_ids_to_tokens(token_ids)


def decode_nllb(tokens, tokenizer: AutoTokenizer) -> str:
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    return tokenizer.decode(token_ids, skip_special_tokens=True)


model_dir = ensure_ct2_model()

print("Lade Tokenizer + CTranslate2 Engine...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
translator = ctranslate2.Translator(
    str(model_dir),
    device="cpu",
    compute_type="int8",
    inter_threads=INTER_THREADS,
    intra_threads=INTRA_THREADS,
)

# Minimaler Test-Datensatz
examples = [
    ("Quantenverschränkung in makroskopischen Systemen", "deu_Latn"),
    ("Efecto Hall cuántico fraccionario en grafeno", "spa_Latn"),
    ("Dynamique des fluides dans les nanocarbones", "fra_Latn"),
    ("Superconduttività ad alta temperatura", "ita_Latn"),
    ("Теория струн и черные дыры", "rus_Cyrl"),
    ("Kinetische Theorie von Plasmen", "deu_Latn"),
    ("Evolución estelar y nucleosíntesis", "spa_Latn"),
    ("Thermodynamique hors équilibre", "fra_Latn"),
]

source_tokens = [encode_nllb(text, src_lang, tokenizer) for text, src_lang in examples]
target_prefix = [[TARGET_LANG]] * len(source_tokens)

results = []
start = time.time()
for i in tqdm(range(0, len(source_tokens), BATCH_SIZE), desc="Uebersetze", unit="batch"):
    batch = source_tokens[i : i + BATCH_SIZE]
    prefix = target_prefix[i : i + BATCH_SIZE]
    results.extend(
        translator.translate_batch(
            batch,
            target_prefix=prefix,
            batch_type="tokens",
            max_batch_size=MAX_BATCH_TOKENS,
            beam_size=1,
        )
    )

translated = []
for out in results:
    hyp = out.hypotheses[0]
    if hyp and hyp[0] == TARGET_LANG:
        hyp = hyp[1:]
    translated.append(decode_nllb(hyp, tokenizer))

elapsed = time.time() - start
print(f"\n✅ Fertig in {elapsed:.2f}s | {len(examples)/elapsed:.2f} Titel/s")

for (src_text, src_lang), tgt_text in zip(examples, translated):
    print(f"\n[{src_lang}] {src_text}\n[eng_Latn] {tgt_text}")

Lade vor-konvertiertes CTranslate2-Modell (einmalig)...


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Download abgeschlossen in 1.3s
Lade Tokenizer + CTranslate2 Engine...


Uebersetze:   0%|          | 0/1 [00:00<?, ?batch/s]


✅ Fertig in 1.35s | 5.91 Titel/s

[deu_Latn] Quantenverschränkung in makroskopischen Systemen
[eng_Latn] Quantum confinement in macroscopic systems is a very important part of the quantum system.

[spa_Latn] Efecto Hall cuántico fraccionario en grafeno
[eng_Latn] Quantum fractional Hall effect in graphene

[fra_Latn] Dynamique des fluides dans les nanocarbones
[eng_Latn] The dynamics of fluids in nanocarbons

[ita_Latn] Superconduttività ad alta temperatura
[eng_Latn] Superconductivity at high temperatures

[rus_Cyrl] Теория струн и черные дыры
[eng_Latn] String theory and black holes.

[deu_Latn] Kinetische Theorie von Plasmen
[eng_Latn] Kinetic theory of plasma

[spa_Latn] Evolución estelar y nucleosíntesis
[eng_Latn] Stellar evolution and nucleosynthesis are the main reasons for this.

[fra_Latn] Thermodynamique hors équilibre
[eng_Latn] Thermodynamics is out of balance.
